## Importar Datos:

In [ ]:
import pandas as pd

In [ ]:
url_salud = 'https://raw.githubusercontent.com/No-Country-simulation/g9-latam-team08/refs/heads/main/Dataset/Transacciones/Dataset_Salud_Financiera_Definido.csv'
url_gastos = 'https://raw.githubusercontent.com/No-Country-simulation/g9-latam-team08/refs/heads/main/Dataset/Perfil/dataset_gastos.csv'

In [ ]:
df_salud = pd.read_csv(url_salud)
df_gastos = pd.read_csv(url_gastos)

In [ ]:
# Análisis EDA: Ver la estructura y los primeros registros
print("============== SALUD FINANCIERA ==============")
print(df_salud.info())
display(df_salud.head(3))

### Diccionario de Datos: Estructura de Salud Financiera

A continuación, se detalla la definición y alcance de cada variable utilizada en el dataset:

| Variable | Tipo / Naturaleza | Descripción y Alcance |
| :--- | :--- | :--- |
| **`id_cliente`** | Identificador | Código único asignado a cada usuario (Ej. USR_001). |
| **`ingreso_mensual_fijo`** | Variable Base | Sumatoria de ingresos recurrentes netos (salario, rentas fijas). |
| **`ingreso_mensual_variable`** | Variable Base | Sumatoria de ingresos no recurrentes (bonos, horas extra). Mide volatilidad. |
| **`ingreso_mensual`** | Calculado | `ingreso_fijo + ingreso_variable`. Ingreso neto total del mes. |
| **`gastos_esenciales_mensuales`** | Variable Base | Gastos vitales: vivienda, salud, supermercado básico. |
| **`gastos_no_escenciales_mensuales`** | Variable Base | Gastos de estilo de vida: ocio, restaurantes, compras prescindibles. |
| **`cuotas_mensuales_deuda`** | Variable Base | Compromisos financieros fijos a pagar en el mes (tarjetas, préstamos). |
| **`gastos_totales_del_mes`** | Calculado | `esenciales + no_esenciales + cuotas_deuda`. Salida total de dinero. |
| **`modalidad_pago_tarjeta`** | Comportamiento | "total", "parcial" o "sin_deuda". Clave para identificar deuda tóxica. |
| **`ahorro_mensual`** | Calculado | `ingreso_mensual - gastos_totales`. Lo que sobra a fin de mes. |
| **`ahorro_previo`** | Variable Base | Capital líquido o inversiones que el usuario ya tenía antes de este mes. |
| **`ahorro_total`** | Calculado | `ahorro_mensual + ahorro_previo`. Liquidez real disponible hoy. |
| **`ratio_ahorro_neto`** | Indicador (Ratio) | `ahorro_mensual / ingreso_mensual`. (El estándar ideal es 20%). |
| **`ratio_endeudamiento_dti`** | Indicador (Ratio) | `cuotas_deuda / ingreso_mensual`. Mide cuánto sueldo está comprometido. |
| **`gastos_esenciales_ratio`** | Indicador (Ratio) | `gastos_esenciales / ingreso_mensual`. |
| **`gastos_estilo_vida_ratio`** | Indicador (Ratio) | `gastos_no_escenciales / ingreso_mensual`. Margen de ajuste ante crisis. |
| **`meses_supervivencia`** | Métrica Resiliencia | `ahorro_total / (gastos_esenciales + cuotas_deuda)`. Meses de supervivencia si los ingresos caen a cero. |
| **`frecuencia_transacciones_ocio`** | Comportamiento | Cantidad de operaciones. Síntoma de "gastos hormiga" y descontrol impulsivo. |
| **`perfil_financiero`** | **Variable Objetivo** | Etiqueta final (Saludable, En Observación, En Riesgo) calculada según ratios y modalidad de pago. |

In [ ]:
print("\n============== DATASET DE GASTOS ==============")
print(df_gastos.info())
display(df_gastos.head(3))


### Diccionario de Datos: Estructura de Gastos

A continuación, se detalla la definición de cada variable utilizada en el dataset de consumos y transacciones:

| Columna | Tipo de Dato | Ejemplo(s) | Descripción |
| :--- | :--- | :--- | :--- |
| **`nombre_tienda`** | Texto (String) | Starbucks, Uber, Falabella, ARBA | Nombre del comercio, entidad o servicio donde se realizó la transacción. |
| **`subcategoria`** | Texto (Categórico) | farmacia, restaurante, combustible | Etiqueta de clasificación "micro" (más granular). Sirve como contexto adicional en el modelo (22 valores posibles). |
| **`monto`** | Numérico (Float) | 4828.97, 59250.57 | Monto exacto de la transacción, expresado en pesos. |
| **`esencial`** | Booleano | True, False | Indica si el gasto se considera esencial (salud, servicios, impuestos) o no esencial (entretenimiento, delivery). |
| **`categoria_principal`** | Texto (Categórico) | Alimentacion, Salud, Transporte | Categoría general del gasto. Es la etiqueta de clasificación "macro" (6 valores posibles). |

<br>

#### 🔗 Relación Jerárquica: Categoría Principal ➔ Subcategoría
Para facilitar el análisis y la agrupación, las categorías macro se subdividen de la siguiente manera:

*   **Alimentación:** `carniceria_y_granja`, `restaurante`, `delivery`, `supermercado`.
*   **Entretenimiento:** `hobbies_y_deportes`, `indumentaria`, `cuidado_personal`, `suscripciones_digitales`.
*   **Finanzas:** `transferencias`, `pago_tarjetas`, `impuestos`.
*   **Hogar:** `mantenimiento_y_muebles`, `alquiler_y_expensas`, `servicios_basicos`.
*   **Salud:** `farmacia`, `atencion_medica`, `cobertura_medica`.
*   **Transporte:** `taxi_y_apps`, `mantenimiento_vehicular`, `transporte_publico`, `combustible`, `peajes`.

## Limpieza de datos:

In [ ]:
import numpy as np

In [ ]:
columnas_numericas = [
    'ingreso_mensual_fijo', 'ingreso_mensual_variable', 'ingreso_mensual',
    'gastos_esenciales_mensuales', 'gastos_no_escenciales_mensuales',
    'cuotas_mensuales_deuda', 'gastos_totales_del_mes', 'ahorro_mensual',
    'ahorro_previo', 'ahorro_total', 'ratio_ahorro_neto',
    'ratio_endeudamiento_dti', 'gastos_esenciales_ratio',
    'gastos_estilo_vida_ratio', 'meses_supervivencia', 'frecuencia_transacciones_ocio'
]

for col in columnas_numericas:
    if df_salud[col].dtype == 'object':
        # Detectamos si la columna tiene porcentajes para ajustarla al final
        es_porcentaje = df_salud[col].astype(str).str.contains('%').any()

        # A. Borramos símbolos de $ y %
        df_salud[col] = df_salud[col].astype(str).str.replace('$', '', regex=False).str.replace('%', '', regex=False)

        # B. Borramos los puntos (.) de los separadores de miles
        df_salud[col] = df_salud[col].str.replace('.', '', regex=False)

        # C. Cambiamos las comas (,) por puntos (.) para el formato decimal
        df_salud[col] = df_salud[col].str.replace(',', '.', regex=False)

        # D. Convertimos a número puro (float)
        df_salud[col] = df_salud[col].astype(float)

        # E. Ajustamos la escala matemática de los porcentajes
        if es_porcentaje:
            df_salud[col] = df_salud[col] / 100


display(df_salud[['ingreso_mensual_fijo', 'ratio_ahorro_neto', 'meses_supervivencia']].head(3))

In [ ]:
print("INICIANDO ESCÁNER DE CALIDAD DE DATOS...\n")

# 1. Chequeo de Nulos
nulos_totales = df_salud.isnull().sum().sum()
if nulos_totales == 0:
    print("✅ Prueba 1 superada: No hay celdas vacías (0 valores nulos).")
else:
    print(f"⚠️ Alerta: Se encontraron {nulos_totales} celdas vacías. Revisa df_salud.isnull().sum()")

# 2. Chequeo de Duplicados
duplicados = df_salud.duplicated().sum()
if duplicados == 0:
    print("✅ Prueba 2 superada: No hay filas duplicadas (500 usuarios únicos).")
else:
    print(f"⚠️ Alerta: Se encontraron {duplicados} filas repetidas. Se recomienda usar df_salud.drop_duplicates()")

# 3. Consistencia de Texto (Limpiamos espacios invisibles y pasamos a minúsculas para estandarizar)
columnas_texto = ['id_cliente', 'modalidad_pago_tarjeta', 'perfil_financiero']
for col in columnas_texto:
    df_salud[col] = df_salud[col].astype(str).str.strip().str.lower()
print("✅ Prueba 3 superada: Textos estandarizados sin espacios ocultos.")



## Preparación de datos

In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
# Se borran letras/espacios y convertimos a número entero puro
df_gastos['id_cliente_limpio'] = df_gastos['id_cliente'].astype(str).str.replace(r'\D', '', regex=True).astype(int)
df_salud['id_cliente_limpio'] = df_salud['id_cliente'].astype(str).str.replace(r'\D', '', regex=True).astype(int)

# 2. Union usando la nueva columna de números puros
df_completo = pd.merge(df_gastos, df_salud, on='id_cliente_limpio', how='inner')

print(f"Fusión exitosa. Transacciones encontradas: {df_completo.shape[0]}")

if df_completo.shape[0] > 0:

    # Buscamos ignorando las mayúsculas usando .str.lower()
    df_riesgo = df_completo[df_completo['perfil_financiero'].str.lower() == 'en riesgo']

    if df_riesgo.shape[0] > 0:
        # Se suman montos por categoría
        gastos_riesgo = df_riesgo.groupby(['categoria_principal', 'subcategoria'])['monto'].sum().reset_index()

        # Gráfico
        fig = px.sunburst(
            gastos_riesgo,
            path=['categoria_principal', 'subcategoria'],
            values='monto',
            title='¿A dónde va el dinero de los perfiles "En Riesgo"?',
            color='categoria_principal',
            template='plotly_white'
        )

        fig.update_layout(margin=dict(t=50, l=0, r=0, b=0), title_x=0.5)
        fig.show()
    else:
        print("El cruce de IDs funcionó, pero no se encontraron usuarios catalogados como 'en riesgo'.")
else:
    print("Siguen sin coincidir los IDs.")

## Definir normas del estado financiero:

In [ ]:
import numpy as np

# 1. Regla: Puntaje por Meses de Supervivencia (Máx 35 pts)
cond_sup = [
    df_salud['meses_supervivencia'] == 0,
    df_salud['meses_supervivencia'] <= 3,
    df_salud['meses_supervivencia'] <= 6,
    df_salud['meses_supervivencia'] > 6
]
val_sup = [0, 15, 25, 35]
df_salud['score_supervivencia'] = np.select(cond_sup, val_sup, default=0)

In [ ]:
# 2. Regla: Puntaje por Ratio de Ahorro (Máx 35 pts)
cond_ahorro = [
    df_salud['ratio_ahorro_neto'] < 0,
    df_salud['ratio_ahorro_neto'] <= 0.10,
    df_salud['ratio_ahorro_neto'] <= 0.20,
    df_salud['ratio_ahorro_neto'] > 0.20
]
val_ahorro = [0, 15, 25, 35]
df_salud['score_ahorro'] = np.select(cond_ahorro, val_ahorro, default=0)

In [ ]:
# 3. Regla: Puntaje por Endeudamiento (DTI) (Máx 30 pts)
cond_deuda = [
    df_salud['ratio_endeudamiento_dti'] > 0.36, # Umbral inaceptable
    df_salud['ratio_endeudamiento_dti'] > 0.20,
    df_salud['ratio_endeudamiento_dti'] >= 0
]
val_deuda = [0, 15, 30]
df_salud['score_endeudamiento'] = np.select(cond_deuda, val_deuda, default=0)

In [ ]:
# 4. Regla: Penalización por Pago Parcial de Tarjeta
penalizacion_parcial = np.where(df_salud['modalidad_pago_tarjeta'] == 'parcial', -15, 0)

In [ ]:
# 5. Cálculo del Score Total (Escala 0 a 100)
df_salud['score_financiero'] = (df_salud['score_supervivencia'] +
                                df_salud['score_ahorro'] +
                                df_salud['score_endeudamiento'] +
                                penalizacion_parcial)

In [ ]:
# Aseguramos que nadie tenga puntaje negativo matemáticamente
df_salud['score_financiero'] = np.where(df_salud['score_financiero'] < 0, 0, df_salud['score_financiero'])

In [ ]:
columnas_ver = ['id_cliente', 'score_supervivencia', 'score_ahorro', 'score_endeudamiento', 'score_financiero', 'perfil_financiero']
display(df_salud[columnas_ver].head(10))

### Registro de Decisiones:

Para la construcción del `score_financiero`, nos basamos en estándares reales de la industria bancaria y reglas de finanzas personales aceptadas globalmente.

**1. Puntaje por Meses de Supervivencia (Máx 35 pts)**
*   **El Estándar:** La recomendación global es mantener un fondo de emergencia equivalente a 3-6 meses de gastos vitales.
*   **Justificación:** Es el capital líquido esencial para soportar un estado de alerta prolongado (pérdida de empleo, crisis médica). 0 meses es un estado crítico (0 pts), de 1 a 3 meses representa un nivel de supervivencia básico (15 pts), y superar los 6 meses demuestra un fuerte blindaje financiero (35 pts).

**2. Puntaje por Ratio de Ahorro (Máx 35 pts)**
*   **El Estándar:** Regla presupuestaria 50/30/20 (20% del ingreso neto destinado al ahorro/inversión).
*   **Justificación:** Ahorrar el 20% marca una salud financiera óptima (35 pts). Entre 11% y 20% es un buen camino con margen de mejora (25 pts). Menos del 10% deja al usuario vulnerable a la inflación (15 pts). Un ratio negativo indica que se gasta más de lo que ingresa, lo que técnicamente representa *desahorro* o quiebra técnica (0 pts).

**3. Puntaje por Endeudamiento / Ratio DTI (Máx 30 pts)**
*   **El Estándar:** "Regla del 36%" (Debt-to-Income Ratio), utilizada como límite máximo en la industria bancaria para el otorgamiento de créditos.
*   **Justificación:** Si la suma de cuotas mensuales supera el 36% de los ingresos, el riesgo de impago se dispara drásticamente (0 pts). Mantener el endeudamiento por debajo del 20% se considera un manejo de crédito sano y responsable (30 pts).

**4. Penalización por Pago Parcial de Tarjeta (-15 pts)**
*   **El Estándar:** Evitar la acumulación del Costo Financiero Total (CFT) por interés rotativo.
*   **Justificación:** Realizar pagos mínimos o parciales genera una bola de nieve de intereses compuestos (la deuda más cara del mercado). Es un síntoma temprano de comportamiento financiero tóxico, por lo que se aplica una penalización directa para alertar sobre la pérdida rápida de liquidez.

**5. Cálculo del Score Total (Escala 0 a 100)**
*   **El Estándar:** Normalización porcentual (base 100) para métricas de usuario.
*   **Justificación:** Sumando los máximos posibles (35 + 35 + 30), el perfil ideal alcanza los 100 puntos. Esta escala es universal, altamente intuitiva (UX) y perfecta para ser consumida visualmente por el Frontend de la aplicación (ej. gráficos de tipo velocímetro).

## Análisis EDA

In [ ]:
!pip install ydata-profiling -q

import pandas as pd
from ydata_profiling import ProfileReport

print("⏳ Generando el reporte automático de EDA... Esto puede demorar unos segundos.")

profile = ProfileReport(
    df_salud,
    title="Análisis Exploratorio Automático - Salud Financiera",
    explorative=True,
)

profile.to_notebook_iframe()

profile.to_file("Reporte_EDA_Salud_Financiera.html")
print("¡Reporte guardado exitosamente!")

### Registro de Análisis Exploratorio de Datos (EDA)

Tras la ejecución del reporte automatizado mediante `ydata-profiling`, se destacan los siguientes insights técnicos que definirán la estrategia de modelado:

**1. Calidad Inmaculada del Dataset**
*   El dataset procesado contiene exactamente 500 observaciones y 23 variables.
*   Se confirmó un **0.0%** de valores nulos (celdas vacías) y un **0.0%** de filas duplicadas. El *Data Pipeline* de limpieza inicial garantizó una base estructuralmente apta para Machine Learning.

**2. Advertencia de Alta Correlación**
*   Se detectaron múltiples alertas por alta correlación ("High correlation").
*   Variables de negocio calculadas (como el `score_financiero`, `score_ahorro` y `ratio_endeudamiento_dti`) presentan una altísima correlación global con las variables numéricas base de las que provienen (como `ahorro_mensual` o `cuotas_mensuales_deuda`).
*   **Decisión Técnica:** En la fase de modelado se deberá realizar una selección de características (*Feature Selection*) para evitar inyectar variables redundantes que generen ruido o sobreajuste (*overfitting*).

**3. La Realidad Financiera de los "Ceros"**
*   El reporte detectó un alto volumen de valores en cero que representan perfiles financieros legítimos, no errores de carga:
    *   **41.0%** de clientes tienen **0** en deuda (usuarios sin pasivos).
    *   **57.2%** registran **0** en ingresos variables (empleados con salario estrictamente fijo).
    *   **29.0%** poseen **0** en ahorro previo (usuarios sin fondo de emergencia histórico).

**4. Confirmación de Desbalanceo Crítico**
*   Se levantó una alerta de clase "Imbalance", indicando que variables objetivo y scores (como el `score_supervivencia`) están altamente desbalanceadas, concentrando un **76.6%** de los datos en una clase mayoritaria.
*   **Decisión Técnica (Próximo Paso):** Esto corrobora matemáticamente la necesidad de aplicar técnicas de sobremuestreo sintético (**SMOTE**) para la clase minoritaria ("En Riesgo") antes de entrenar el algoritmo, evitando así que el modelo predictivo quede sesgado hacia la clase mayoritaria.

## Tratamiento de Desbalanceo

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from collections import Counter

In [ ]:
# 1. Separar 'X' (características predictivas) e 'y' (variable objetivo)
# Tiramos el 'id_cliente' porque es texto y no predice nada
X = df_salud.drop(columns=['id_cliente', 'perfil_financiero'])
y = df_salud['perfil_financiero']

In [ ]:
# 2. Transformación de Categóricas
# Transformamos la columna de texto 'modalidad_pago_tarjeta' en columnas numéricas (0 y 1)
X = pd.get_dummies(X, columns=['modalidad_pago_tarjeta'], drop_first=True)

In [ ]:
print(f"Distribución ORIGINAL (Desbalanceada):")
for perfil, cantidad in Counter(y).items():
    print(f"   - {perfil}: {cantidad} usuarios")

In [ ]:
# 3. Aplicar SMOTE para generar los datos sintéticos
smote = SMOTE(random_state=42)
X_balanceado, y_balanceado = smote.fit_resample(X, y)

print("\n Aplicando sobremuestreo sintético ...")
print(f"📈 Distribución NUEVA (Balanceada):")
for perfil, cantidad in Counter(y_balanceado).items():
    print(f"   - {perfil}: {cantidad} usuarios")

In [ ]:
# 4. Reconstruir el DataFrame final listo para Machine Learning
df_ml_ready = pd.concat([X_balanceado, y_balanceado], axis=1)
print(f"\n✅ ¡Éxito! Tu dataset pasó de {len(df_salud)} a {len(df_ml_ready)} filas perfectamente equilibradas.")

## Entrenamiento del modelo:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# 1. Selección de Variables
columnas_elegidas = ['meses_supervivencia', 'score_supervivencia',
                     'score_ahorro', 'score_endeudamiento', 'score_financiero']

X_final = df_ml_ready[columnas_elegidas]
y_final = df_ml_ready['perfil_financiero']

In [ ]:
# 2. División de los datos (80% para estudiar, 20% para el examen)
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

print(f"📚 Datos para entrenar: {len(X_train)} clientes")
print(f"📝 Datos para el examen: {len(X_test)} clientes\n")

In [ ]:
# 3. Creación y Entrenamiento del Modelo
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
print("Entrenando el algoritmo... (aprendiendo patrones financieros)")
modelo_rf.fit(X_train, y_train)

In [ ]:
# 4. El Examen (Evaluación del Modelo)
predicciones = modelo_rf.predict(X_test)
precision = accuracy_score(y_test, predicciones)

print(f"¡Entrenamiento completado!")
print(f"Precisión Global: {precision * 100:.2f}%\n")

print("Reporte Detallado de Clasificación:")
print(classification_report(y_test, predicciones))

### Conclusión:

Para la etapa de modelado, se implementó un algoritmo **Random Forest Classifier (Bosque Aleatorio)**. La estrategia técnica se fundamentó en dos decisiones clave tomadas a partir de los hallazgos del EDA:

1.  **Selección de Variables (*Feature Selection*):** Se entrenó al modelo utilizando exclusivamente los scores financieros calculados y los `meses_supervivencia`.
2.  **Uso de Datos Balanceados:** El algoritmo fue alimentado con el conjunto de datos nivelado matemáticamente mediante la técnica **SMOTE**, lo que eliminó el sesgo inicial hacia la clase mayoritaria.

**Resultados del examen (Conjunto de Test - 20% de los datos):**

El desempeño del modelo superó las expectativas con una **Precisión Global del 96.21%**. Destacan los siguientes hitos:
*   **Detección Perfecta del Riesgo:** El modelo alcanzó una precisión y exhaustividad de **1.00** para el perfil "En Riesgo". Esto significa que logró identificar al 100% de los clientes vulnerables sin generar falsos positivos.
*   **Capacidad de Generalización:** Al evaluar al modelo con 132 registros nunca antes vistos, demostró no haber memorizado los datos, sino haber aprendido genuinamente las reglas de clasificación financiera.



## .pkl

In [ ]:
import joblib
from google.colab import files

In [ ]:
joblib.dump(modelo_rf, 'modelo_riesgo_financiero.pkl')

In [ ]:
files.download('modelo_riesgo_financiero.pkl')

In [ ]:
import pandas as pd
import sklearn
import imblearn
import joblib

print("Versiones utilizadas:")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-Learn: {sklearn.__version__}")
print(f"Imbalanced-Learn (SMOTE): {imblearn.__version__}")
print(f"Joblib: {joblib.__version__}")

# Modelo de clasificación de categorías de transacciones


El objetivo del segundo modelo es predecir `categoria_principal` a partir de `nombre_tienda`, `subcategoria` y `esencial`.

La arquitectura se inspira en el enfoque **Two-headed DragoNet** de Busson et al. (2023): generar representaciones contextuales de información textual y fusionarlas antes de clasificar. La implementación de este proyecto es una adaptación simplificada: utiliza una sola salida para `categoria_principal`, en lugar de la clasificación jerárquica macro/micro del trabajo original.

## EDA — Dataset de transacciones

El EDA se realiza con el mismo enfoque automatizado utilizado para el modelo de riesgo: `ydata-profiling`. Además del reporte automático, se documentan las observaciones que afectan directamente la selección de características.

In [ ]:
# EDA del dataset utilizado por el modelo de categorías
!pip install ydata-profiling -q

from ydata_profiling import ProfileReport

df_categoria = df_gastos.copy()

print("============== DATASET PARA CLASIFICACIÓN DE CATEGORÍAS ==============")
print(df_categoria.info())
display(df_categoria.head(5))

print("\nDimensiones:", df_categoria.shape)
print("\nValores nulos por columna:")
display(df_categoria.isnull().sum())

print("\nFilas duplicadas:", df_categoria.duplicated().sum())

print("\nDistribución de la variable objetivo:")
display(df_categoria['categoria_principal'].value_counts().sort_index())

print("\nDistribución porcentual de la variable objetivo:")
display(
    (df_categoria['categoria_principal'].value_counts(normalize=True)
     .mul(100)
     .round(2)
     .sort_index()
     .rename("porcentaje"))
)

profile_categoria = ProfileReport(
    df_categoria,
    title="Análisis Exploratorio Automático - Clasificación de Categorías",
    explorative=True,
)

profile_categoria.to_notebook_iframe()
profile_categoria.to_file("Reporte_EDA_Modelo_Categoria.html")

print("Reporte EDA guardado como: Reporte_EDA_Modelo_Categoria.html")

### Lectura del EDA

El dataset de transacciones contiene 2000 registros y 7 columnas, sin valores nulos ni filas duplicadas según el análisis documentado del proyecto.

Las variables relevantes son:

- `nombre_tienda`: texto del comercio o servicio.
- `subcategoria`: categoría más específica de la transacción.
- `esencial`: indica si el gasto es esencial.
- `categoria_principal`: variable objetivo, con 6 categorías.

El EDA también muestra dos señales importantes para el modelado:

1. `subcategoria` presenta una relación muy fuerte con `categoria_principal`.
2. `esencial` también presenta una relación fuerte con la categoría objetivo.

Por otro lado, `id_cliente` funciona como identificador, `monto` presenta valores únicos por fila y `metodo_pago` no muestra una señal suficiente para justificar su inclusión. Por estas razones, esas tres variables se excluyen del modelo.

Una consecuencia importante del EDA es que el modelo puede alcanzar un desempeño extremadamente alto porque `subcategoria` prácticamente determina `categoria_principal` en este dataset. Por tanto, un 100% de F1 debe interpretarse junto con esta característica del conjunto de datos y no como evidencia de que el modelo tendrá necesariamente el mismo rendimiento sobre datos externos.

## Preprocesamiento


1. `esencial` se normaliza a `float32`, usando `1` para verdadero y `0` para falso.
2. `nombre_tienda` y `subcategoria` se procesan mediante `TextVectorization`.
3. El vocabulario del vectorizador se ajusta **únicamente con el conjunto de entrenamiento**, evitando data leakage.
4. `categoria_principal` se transforma mediante `LabelEncoder`.
5. Se utiliza una división 80/20 con `random_state=42`.

No se agregan variables derivadas nuevas. `esencial` se incorpora como entrada numérica adicional, mientras que los dos campos de texto pasan por los bloques Transformer.

In [ ]:
import sys
import numpy as np
import tensorflow as tf
import sklearn
import pickle

from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Dataset ya cargado por el notebook principal
X_nombre_cat = df_categoria['nombre_tienda'].astype(str).values
X_subcategoria_cat = df_categoria['subcategoria'].astype(str).values

def normalizar_esencial_categoria(serie):
    # Convierte si/no, true/false y 1/0 a float32 0/1.
    serie = serie.astype(str).str.strip().str.lower()
    mapa = {
        'si': 1, 'sí': 1, 'true': 1, '1': 1,
        'no': 0, 'false': 0, '0': 0
    }
    return serie.map(mapa).astype('float32')

X_esencial_cat = normalizar_esencial_categoria(
    df_categoria['esencial']
).values.reshape(-1, 1)

label_encoder_categoria = LabelEncoder()
y_categoria = label_encoder_categoria.fit_transform(
    df_categoria['categoria_principal']
)

num_clases_categoria = len(label_encoder_categoria.classes_)

print("Clases de categoria_principal:")
print(list(label_encoder_categoria.classes_))

(
    X_nom_train_cat, X_nom_test_cat,
    X_sub_train_cat, X_sub_test_cat,
    X_esc_train_cat, X_esc_test_cat,
    y_train_cat, y_test_cat
) = train_test_split(
    X_nombre_cat,
    X_subcategoria_cat,
    X_esencial_cat,
    y_categoria,
    test_size=0.2,
    random_state=42
)

print(f"\nEntrenamiento: {len(y_train_cat)} registros")
print(f"Test: {len(y_test_cat)} registros")

## Vectorización de texto

`TextVectorization` realiza la conversión de texto a secuencias de índices. Se utiliza un vocabulario máximo de 5000 tokens y secuencias de longitud 5.

El `adapt()` se ejecuta únicamente sobre los datos de entrenamiento. Esto es importante porque ajustar el vocabulario con todo el dataset antes de separar train/test introduciría información del conjunto de prueba en el proceso de preparación.

In [ ]:
max_tokens_categoria = 5000
sequence_length_categoria = 5

vectorize_layer_categoria = layers.TextVectorization(
    max_tokens=max_tokens_categoria,
    output_mode='int',
    output_sequence_length=sequence_length_categoria
)

# El vocabulario se aprende solo desde train
vectorize_layer_categoria.adapt(
    np.concatenate((X_nom_train_cat, X_sub_train_cat))
)

print(f"Tamaño del vocabulario generado: {len(vectorize_layer_categoria.get_vocabulary())}")
print("Primeros tokens:")
print(vectorize_layer_categoria.get_vocabulary()[:20])

## Arquitectura — Transformer + Context Fusion


**nombre_tienda → Embedding → Transformer → Pooling**  
**subcategoria → Embedding → Transformer → Pooling**  
**esencial → entrada numérica**

Después, las tres representaciones se concatenan mediante una capa de **Context Fusion** y pasan por una capa densa antes de la salida `softmax`.

La motivación viene del trabajo de Busson et al.: combinar diferentes fuentes de contexto puede producir una representación más útil que clasificar utilizando únicamente el nombre del comercio. El proyecto adapta esta idea a sus propias columnas y a una sola categoría objetivo.

In [ ]:
def transformer_encoder_categoria(
    inputs,
    embed_dim,
    num_heads,
    ff_dim,
    dropout_rate=0.1
):
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim
    )(inputs, inputs)

    attn_output = layers.Dropout(dropout_rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(
        inputs + attn_output
    )

    ffn_output = layers.Dense(
        ff_dim,
        activation="relu"
    )(out1)

    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(dropout_rate)(ffn_output)

    return layers.LayerNormalization(epsilon=1e-6)(
        out1 + ffn_output
    )

embed_dim_categoria = 64
num_heads_categoria = 2
ff_dim_categoria = 64

input_nombre_categoria = layers.Input(
    shape=(1,),
    dtype=tf.string,
    name='input_nombre'
)

input_subcategoria_categoria = layers.Input(
    shape=(1,),
    dtype=tf.string,
    name='input_subcategoria'
)

input_esencial_categoria = layers.Input(
    shape=(1,),
    dtype=tf.float32,
    name='input_esencial'
)

emb_nombre_categoria = layers.Embedding(
    input_dim=max_tokens_categoria,
    output_dim=embed_dim_categoria
)(
    vectorize_layer_categoria(input_nombre_categoria)
)

emb_subcategoria_categoria = layers.Embedding(
    input_dim=max_tokens_categoria,
    output_dim=embed_dim_categoria
)(
    vectorize_layer_categoria(input_subcategoria_categoria)
)

trans_nombre_categoria = transformer_encoder_categoria(
    emb_nombre_categoria,
    embed_dim_categoria,
    num_heads_categoria,
    ff_dim_categoria
)

trans_subcategoria_categoria = transformer_encoder_categoria(
    emb_subcategoria_categoria,
    embed_dim_categoria,
    num_heads_categoria,
    ff_dim_categoria
)

pool_nombre_categoria = layers.GlobalAveragePooling1D()(
    trans_nombre_categoria
)

pool_subcategoria_categoria = layers.GlobalAveragePooling1D()(
    trans_subcategoria_categoria
)

context_fusion_categoria = layers.Concatenate(
    name='context_fusion'
)([
    pool_nombre_categoria,
    pool_subcategoria_categoria,
    input_esencial_categoria
])

context_fusion_categoria = layers.Dense(
    128,
    activation='relu'
)(context_fusion_categoria)

context_fusion_categoria = layers.Dropout(0.2)(
    context_fusion_categoria
)

output_categoria = layers.Dense(
    num_clases_categoria,
    activation='softmax',
    name='salida_categoria'
)(context_fusion_categoria)

model_categoria = Model(
    inputs=[
        input_nombre_categoria,
        input_subcategoria_categoria,
        input_esencial_categoria
    ],
    outputs=output_categoria,
    name='modelo_categoria'
)

model_categoria.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_categoria.summary()

## Entrenamiento


- Optimizador: **Adam**
- Learning rate: **0.001**
- Función de pérdida: `sparse_categorical_crossentropy`
- Épocas: **10**
- `batch_size`: **32**

La evaluación se realiza sobre el 20% reservado como test.

In [ ]:
history_categoria = model_categoria.fit(
    {
        'input_nombre': X_nom_train_cat,
        'input_subcategoria': X_sub_train_cat,
        'input_esencial': X_esc_train_cat
    },
    y_train_cat,
    validation_data=(
        {
            'input_nombre': X_nom_test_cat,
            'input_subcategoria': X_sub_test_cat,
            'input_esencial': X_esc_test_cat
        },
        y_test_cat
    ),
    epochs=10,
    batch_size=32
)

## Evaluación — Precision, Recall y F1

Además del `accuracy`, se utiliza `classification_report` para observar el rendimiento individual de las seis categorías.

Esto es importante porque una métrica global puede ocultar problemas en clases específicas. En este dataset, sin embargo, la distribución de `categoria_principal` es relativamente equilibrada, por lo que el accuracy resulta útil como métrica general y las métricas por clase permiten confirmar que el rendimiento no depende de una sola categoría.

In [ ]:
pred_probs_categoria = model_categoria.predict(
    {
        'input_nombre': X_nom_test_cat,
        'input_subcategoria': X_sub_test_cat,
        'input_esencial': X_esc_test_cat
    },
    verbose=0
)

y_pred_categoria = np.argmax(
    pred_probs_categoria,
    axis=1
)

accuracy_categoria = accuracy_score(
    y_test_cat,
    y_pred_categoria
)

print(f"Accuracy en test: {accuracy_categoria:.2%}\n")
print("Precision / Recall / F1-score por clase:\n")

print(
    classification_report(
        y_test_cat,
        y_pred_categoria,
        target_names=label_encoder_categoria.classes_,
        digits=3
    )
)

### Interpretación del resultado


Este resultado es coherente con el EDA: `subcategoria` mantiene una relación prácticamente determinista con `categoria_principal`. En otras palabras, al entregar `subcategoria` al modelo, gran parte de la información necesaria para determinar la categoría principal ya está presente.

Por esta razón, el resultado debe interpretarse como un desempeño sobre **este dataset específico**. No debe extrapolarse automáticamente a transacciones externas con subcategorías ambiguas, nuevas o inconsistentes.

## Serialización del modelo de categorías

El modelo de categorías se guarda en formato nativo **Keras (`.keras`)**. Los objetos auxiliares que no pertenecen al modelo Keras se guardan por separado en un archivo **`.pkl`**.

El `.pkl` contiene:

- `LabelEncoder`, necesario para recuperar los nombres de las categorías.
- vocabulario de `TextVectorization`.
- versiones de Python, TensorFlow, Keras, scikit-learn, NumPy y Pandas utilizadas durante el entrenamiento.

Esta separación permite reconstruir correctamente el procesamiento utilizado por el modelo al momento de realizar inferencias.

In [ ]:
import keras

# 1. Modelo neuronal de categorías
model_categoria.save("modelo_categoria_full.keras")

# 2. Artefactos auxiliares
config_vectorizador_categoria = {
    'vocabulario': vectorize_layer_categoria.get_vocabulary()
}

versiones_librerias_categoria = {
    'python': sys.version.split()[0],
    'tensorflow': tf.__version__,
    'keras': keras.__version__,
    'scikit-learn': sklearn.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__
}

artefactos_categoria = {
    'label_encoder': label_encoder_categoria,
    'config_vectorizador': config_vectorizador_categoria,
    'versiones_librerias': versiones_librerias_categoria
}

with open("artefactos_categoria.pkl", "wb") as f:
    pickle.dump(artefactos_categoria, f)

print("Archivos generados:")
print(" - modelo_categoria_full.keras")
print(" - artefactos_categoria.pkl")

print("\nVersiones utilizadas:")
for libreria, version in versiones_librerias_categoria.items():
    print(f"  {libreria}: {version}")

## Artefactos finales del notebook integrado

Al ejecutar el notebook completo se obtienen los artefactos de ambos modelos:

### Modelo de riesgo financiero
- `modelo_riesgo_financiero.pkl`

### Modelo de categoría de transacciones
- `modelo_categoria_full.keras`
- `artefactos_categoria.pkl`

## Referencias utilizadas

- Busson, A. J. G. et al. (2023). *Hierarchical Classification of Financial Transactions Through Context-Fusion of Transformer-based Embeddings and Taxonomy-aware Attention Layer*. arXiv:2312.07730.